# Crypto and Stablecoin Liquidity Pulse

This notebook uses real public crypto and DeFi data. It is descriptive and not investment advice.

Sources:

- CoinGecko market chart for BTC/ETH price history;
- DeFiLlama stablecoin data for liquidity context.

No synthetic fallback is used.


In [1]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd

# Prefer the checkout when this notebook is run inside the repository.
repo_root = Path.cwd()
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]:
    if (candidate / "src" / "detime").exists():
        repo_root = candidate
        break
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))

from examples.hot_trends.data import (
    HotTrendDataError,
    append_real_snapshot,
    build_arxiv_monthly_counts,
    fetch_coingecko_market_chart,
    fetch_defillama_stablecoin_chains,
    fetch_github_repo_metadata,
    fetch_github_stargazers,
    fetch_huggingface_models,
    fetch_wikipedia_pageviews,
    source_audit_table,
)
from examples.hot_trends.decomposition import (
    component_summary,
    decompose_table,
    editorial_priority,
    residual_event_table,
)
from examples.hot_trends.scoring import article_language_guardrails

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

CACHE_DIR = repo_root / "examples" / "hot_trends" / "cache"
OUTPUT_DIR = repo_root / "examples" / "hot_trends" / "outputs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_table(df, name):
    path = OUTPUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    relative_path = path.relative_to(repo_root).as_posix()
    print(f"saved: {relative_path}")


## 1. Fetch real BTC and ETH market charts


In [2]:
coins = ["bitcoin", "ethereum"]
frames = [fetch_coingecko_market_chart(c, days=365) for c in coins]
prices = pd.concat(frames, ignore_index=True)
prices.head(20)


,date,coin_id,price,source,data_quality
0,2025-05-23,bitcoin,111560.356938,CoinGecko API,live_public_api_no_synthetic_fallback
1,2025-05-24,bitcoin,107216.668569,CoinGecko API,live_public_api_no_synthetic_fallback
2,2025-05-25,bitcoin,107831.363744,CoinGecko API,live_public_api_no_synthetic_fallback
3,2025-05-26,bitcoin,108861.810377,CoinGecko API,live_public_api_no_synthetic_fallback
4,2025-05-27,bitcoin,109377.715133,CoinGecko API,live_public_api_no_synthetic_fallback
5,2025-05-28,bitcoin,109068.456949,CoinGecko API,live_public_api_no_synthetic_fallback
6,2025-05-29,bitcoin,107838.184311,CoinGecko API,live_public_api_no_synthetic_fallback
7,2025-05-30,bitcoin,105745.416604,CoinGecko API,live_public_api_no_synthetic_fallback
8,2025-05-31,bitcoin,104010.919562,CoinGecko API,live_public_api_no_synthetic_fallback
9,2025-06-01,bitcoin,104687.507429,CoinGecko API,live_public_api_no_synthetic_fallback


## 2. Audit real price table


In [3]:
audit = source_audit_table(prices, value_col="price", entity_col="coin_id", time_col="date")
audit


,series,first_timestamp,last_timestamp,observations,missing_ratio,min_value,max_value
0,bitcoin,2025-05-23 00:00:00,2026-05-22 00:00:00,366,0.0,62853.690384,124773.508231
1,ethereum,2025-05-23 00:00:00,2026-05-22 00:00:00,366,0.0,1820.569322,4829.225542


## 3. Decompose crypto price series


In [4]:
components = decompose_table(prices, entity_col="coin_id", time_col="date", value_col="price", method="MA_BASELINE", period=7, trend_window=21, transform="log")
summary = editorial_priority(component_summary(components, entity_col="coin_id", time_col="date"), entity_col="coin_id")
summary


,coin_id,observations,first_timestamp,last_timestamp,trend_last,trend_slope_per_step,cycle_strength_proxy,residual_std,max_abs_residual_z,method,trend_slope_per_step_rank_pct,cycle_strength_proxy_rank_pct,max_abs_residual_z_rank_pct,editorial_priority_score
1,ethereum,366,2025-05-23 00:00:00,2026-05-22 00:00:00,4.025978,-0.001398,-2.730281,0.523392,64.025261,MA_BASELINE,1.0,1.0,0.5,0.775
0,bitcoin,366,2025-05-23 00:00:00,2026-05-22 00:00:00,5.902171,-0.001546,-14.441693,0.770783,125.690311,MA_BASELINE,0.5,0.5,1.0,0.725


## 4. Residual events


In [5]:
events = residual_event_table(components, entity_col="coin_id", time_col="date", top_n=20)
events


,date,coin_id,observed,trend,season,residual,residual_z,abs_residual_z,method
0,2025-05-23,bitcoin,11.622321,6.068095,0.080851,5.473375,125.690311,125.690311,MA_BASELINE
1,2026-05-22,bitcoin,11.258631,5.902171,0.079332,5.277128,121.164017,121.164017,MA_BASELINE
2,2025-05-24,bitcoin,11.582607,6.619053,0.079332,4.884223,112.101945,112.101945,MA_BASELINE
3,2026-05-22,bitcoin,11.248762,6.440796,0.080851,4.727115,108.478369,108.478369,MA_BASELINE
4,2025-05-25,bitcoin,11.588324,7.169807,-0.031238,4.449755,102.081260,102.081260,MA_BASELINE
5,2026-05-21,bitcoin,11.257516,6.979665,-0.022160,4.300011,98.627530,98.627530,MA_BASELINE
6,2025-05-26,bitcoin,11.597835,7.720280,-0.039326,3.916880,89.790886,89.790886,MA_BASELINE
7,2026-05-20,bitcoin,11.249075,7.517676,-0.036118,3.767517,86.345941,86.345941,MA_BASELINE
8,2025-05-27,bitcoin,11.602562,8.269294,-0.034421,3.367690,77.124206,77.124206,MA_BASELINE
9,2026-05-19,bitcoin,11.250940,8.055397,-0.034421,3.229965,73.947679,73.947679,MA_BASELINE


## 5. Fetch stablecoin context from DeFiLlama

This cell may require schema adjustment if the DeFiLlama stablecoin endpoint changes. It fails explicitly rather than using a fake table.


In [6]:
stable_chains = fetch_defillama_stablecoin_chains()
stable_chains.head(20)


,totalCirculatingUSD,name
0,{'peggedUSD': 7236556.082084637},Manta
1,{'peggedUSD': 475508.41547208483},ThunderCore
2,{'peggedUSD': 39708614.396872304},Movement
3,{'peggedUSD': 32730.435007712756},Shiden
4,{'peggedUSD': 138869.46358001293},Corn
5,{'peggedUSD': 345626480.26076144},Starknet
6,"{'peggedUSD': 90320020359.47278, 'peggedREAL':...",Tron
7,{'peggedUSD': 3534550.2012105375},CORE
8,{'peggedUSD': 317377.32084033196},ApeChain
9,"{'peggedUSD': 3776701.613223316, 'peggedSGD': ...",Zilliqa


## 6. Publication guardrails


In [7]:
guardrails = article_language_guardrails()
guardrails


,unsafe,safer
0,This trend predicts the next price move.,This trend summarizes the observed public seri...
1,This model is better because it has more downl...,Downloads are a public adoption proxy and shou...
2,This repo is winning because stars are rising.,"Star velocity measures developer attention, no..."
3,This pageview spike proves importance.,Pageviews measure public attention during the ...
4,This residual is a buy signal.,This residual marks an event-like deviation fr...


In [8]:
save_table(audit, "06_crypto_price_audit")
save_table(summary, "06_crypto_price_summary")
save_table(events, "06_crypto_price_residual_events")
save_table(stable_chains, "06_defillama_stablecoin_context")
save_table(guardrails, "06_crypto_guardrails")


saved: examples/hot_trends/outputs/06_crypto_price_audit.csv
saved: examples/hot_trends/outputs/06_crypto_price_summary.csv
saved: examples/hot_trends/outputs/06_crypto_price_residual_events.csv
saved: examples/hot_trends/outputs/06_defillama_stablecoin_context.csv
saved: examples/hot_trends/outputs/06_crypto_guardrails.csv
